In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/acecod3z/Flyrankinternship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Rule

I prioritize pages that have high search volume, low CTR compared to their average position, and old content.

Higher score means the page is a stronger candidate for content refresh.

### Reason Codes

CTR_LOW

STALE_CONTENT

HIGH_VOLUME

CTR_LOW_STALE

LOW_PRIORITY

In [ ]:
import pandas as pd
import numpy as np

# ----------------------------
# Rule Conditions
# ----------------------------

high_volume = df["search_volume"] >= 100
low_ctr = df["ctr"] < 1
poor_position = df["avg_position"] > 20

# ----------------------------
# Transparent Baseline Score
# ----------------------------

df["score"] = (
    high_volume.astype(int)
    + low_ctr.astype(int)
    + poor_position.astype(int)
)

# ----------------------------
# Reason Codes
# ----------------------------

conditions = [
    high_volume & low_ctr & poor_position,
    high_volume & low_ctr,
    low_ctr,
    poor_position
]

choices = [
    "HIGH_VOLUME_LOW_CTR_POOR_POSITION",
    "HIGH_VOLUME_LOW_CTR",
    "LOW_CTR",
    "POOR_POSITION"
]

df["reason_code"] = np.select(
    conditions,
    choices,
    default="LOW_PRIORITY"
)

# ----------------------------
# Display Results
# ----------------------------

print("=" * 60)
print("Score Distribution")
print(df["score"].value_counts().sort_index())

print("\n" + "=" * 60)
print("Reason Code Distribution")
print(df["reason_code"].value_counts())

print("\n" + "=" * 60)
print("Sample Output")

display(
    df[
        [
            "content_id",
            "search_volume",
            "ctr",
            "avg_position",
            "score",
            "reason_code"
        ]
    ].head(10)
)

Score Distribution
score
0     1469
1    18379
2     8956
3     1196
Name: count, dtype: int64

Reason Code Distribution
reason_code
LOW_CTR                              25306
HIGH_VOLUME_LOW_CTR                   1789
LOW_PRIORITY                          1524
HIGH_VOLUME_LOW_CTR_POOR_POSITION     1196
POOR_POSITION                          185
Name: count, dtype: int64

Sample Output


,content_id,search_volume,ctr,avg_position,score,reason_code
0,content_304f48230142,10.0,0.76,10.6,1,LOW_CTR
1,content_a1fb4e703a9e,90.0,0.05,20.3,2,LOW_CTR
2,content_9aa793d4d895,0.0,0.09,36.5,2,LOW_CTR
3,content_331d6c4de07b,10.0,0.49,6.2,1,LOW_CTR
4,content_d99b7a2d90ca,0.0,0.13,44.0,2,LOW_CTR
5,content_d4084a4bc775,720.0,0.03,8.5,2,HIGH_VOLUME_LOW_CTR
6,content_9a34b442b552,0.0,0.00,7.0,1,LOW_CTR
7,content_a63219c6e95a,590.0,0.06,21.2,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION
8,content_5e6c160719bc,0.0,0.09,46.0,2,LOW_CTR
9,content_c27558df2b0c,0.0,0.16,4.9,1,LOW_CTR


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# ----------------------------
# Action Label
# ----------------------------

df["action_label"] = "Review Content"

# ----------------------------
# Rank Pages
# ----------------------------

ranked = (
    df.sort_values(
        by="score",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top 20 Ranked Pages")

display(
    ranked[
        [
            "content_id",
            "score",
            "reason_code",
            "action_label"
        ]
    ].head(20)
)

# ----------------------------
# Save CSV
# ----------------------------

import os

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully:")
print("work/outputs/baseline_action_score.csv")

Top 20 Ranked Pages


,content_id,score,reason_code,action_label
0,content_236a15b22de7,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
1,content_a11501e6e45d,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
2,content_66a57f9f17f8,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
3,content_d5e1b986b5fa,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
4,content_9dbcea4d28aa,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
5,content_36e5d52b570c,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
6,content_c3dd69918c8c,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
7,content_cbf712f3775d,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
8,content_b56f9bbafbf9,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content
9,content_70425373a183,3,HIGH_VOLUME_LOW_CTR_POOR_POSITION,Review Content


CSV saved successfully:
work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = ranked.head(20)

print("=" * 80)
print("TOP-20 REVIEW")
print("=" * 80)

for i, row in top20.iterrows():

    print(f"\nRank {i+1}")
    print(f"Content ID : {row['content_id']}")
    print(f"Action     : {row['action_label']}")
    print(f"Reason     : {row['reason_code']}")
    print("Confidence : Medium")
    print("Wrong if   : The page is seasonal, recently updated, or has naturally low CTR due to highly competitive search results.")

TOP-20 REVIEW

Rank 1
Content ID : content_236a15b22de7
Action     : Review Content
Reason     : HIGH_VOLUME_LOW_CTR_POOR_POSITION
Confidence : Medium
Wrong if   : The page is seasonal, recently updated, or has naturally low CTR due to highly competitive search results.

Rank 2
Content ID : content_a11501e6e45d
Action     : Review Content
Reason     : HIGH_VOLUME_LOW_CTR_POOR_POSITION
Confidence : Medium
Wrong if   : The page is seasonal, recently updated, or has naturally low CTR due to highly competitive search results.

Rank 3
Content ID : content_66a57f9f17f8
Action     : Review Content
Reason     : HIGH_VOLUME_LOW_CTR_POOR_POSITION
Confidence : Medium
Wrong if   : The page is seasonal, recently updated, or has naturally low CTR due to highly competitive search results.

Rank 4
Content ID : content_d5e1b986b5fa
Action     : Review Content
Reason     : HIGH_VOLUME_LOW_CTR_POOR_POSITION
Confidence : Medium
Wrong if   : The page is seasonal, recently updated, or has naturally low CTR 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
print("=" * 80)
print("WEAK PICKS")
print("=" * 80)

print("""
1. Pages with very low search volume may have unstable CTR.

2. Some pages naturally rank lower because of high competition.

3. Seasonal pages may temporarily have poor CTR.

4. Informational pages usually have different click behaviour than transactional pages.
""")

print("=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

print("""
✓ content_id was NOT used as a feature.

✓ client_id was NOT used as a feature.

✓ trend_pct was NOT used.

✓ trend_direction was NOT used.

✓ is_declining_label was NOT used.

✓ Only observable current metrics were used.
""")

WEAK PICKS

1. Pages with very low search volume may have unstable CTR.

2. Some pages naturally rank lower because of high competition.

3. Seasonal pages may temporarily have poor CTR.

4. Informational pages usually have different click behaviour than transactional pages.

LEAKAGE CHECK

✓ content_id was NOT used as a feature.

✓ client_id was NOT used as a feature.

✓ trend_pct was NOT used.

✓ trend_direction was NOT used.

✓ is_declining_label was NOT used.

✓ Only observable current metrics were used.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.